# How the floor is defined: a level ladder and one alternative estimator for the floor clip, 49 ROIs x 6 clicks, `mo_pad61`, post-D11

**Pre-registration.** This cell was written before any search ran (2026-10-02). It fixes the arms, the metrics, the
margins and the decision rules, and nothing below changes them. The reading is the last cell.

## Why this experiment

The floor clip `hem_c = np.maximum(hem, b)` was validated with one floor: `b_otsu`, the lower threshold of a
3-class multi-Otsu over the whole ROI. That run is `tm_border_floor_49roi_6seed.ipynb`, audited and judged in
`tm_border_floor_arm_verdicts.ipynb`.
- **The product estimate.** `A3 - A1` = +3.30 pts at P@10 and +2.62 at P@20 (`tm_score`). The pool gains about +1.5
  mitoses per click.
- **Why the clip works.** Under `TM_CCOEFF`, matching on `max(hem, b)` is exactly matching on `max(hem - b, 0)`. The
  clip caps the ring-lightness half of the mass term and keeps the box-darkness half and the shape term.

That argument says *a* floor near the background level should work. It does not say *this* floor. Before adoption,
three questions must have answers:
1. **Is the gain on a plateau in the floor's level?** If it is knife-edge, any estimator error on another lab's slides
   would lose it. The plateau width is the quantity that carries to other labs and scanners, because any adaptive
   estimator will land somewhere off the ideal there. Stain and scanner shift itself is a separate test, not this one.
2. **Does the estimator matter beyond its level?** A uniform level shift is covered by the ladder. This asks whether
   a floor that departs unevenly from `b_otsu`, click by click, still works.
3. **Where does it start to hurt?** That happens when the floor reaches the seed's own chromatin.

## What is fixed

Everything else is the recorded `A3` search:
- the recorded clicks and refinement boxes, which are computed on the unclipped channel;
- the `mo_pad61` template cut from the clipped channel, the D11 51 px blank filled with `channel.min()` (which is `b`),
  and the search on the clipped channel;
- `cv2.TM_CCOEFF` (D1), `MAX_PEAKS = 100` (D9), and NMS at `ev.radius_px(mpp)` (D7);
- **the corrected harness**: the 30 px edge band invalid and floored, as in `A1` and `A3`;
- ranking by **`tm_score`**, the production ranker (D5). Chromatin od is computed by the copied code but not analysed.

## Arms

| arm | floor `b` | role |
|---|---|---|
| `A1` | none | no clip. The recorded run is used, and it is re-searched on click 0 of every ROI as a parity gate. |
| `L025` | 0.25 x `b_otsu` | ladder: is a near-glass floor enough? |
| `L050` | 0.5 x `b_otsu` | ladder: the plateau's lower neighbour |
| `A3` | 1.0 x `b_otsu` | the validated floor; re-searched on all 294 clicks and must equal the recorded run exactly |
| `L150` | 1.5 x `b_otsu` | ladder: the plateau's upper neighbour |
| `L200` | 2.0 x `b_otsu` | ladder: the upper edge, where the floor nears the seed's chromatin |
| `ALT` | one alternative estimator, chosen below **before any search** | does the estimator matter beyond its level? |

**How `ALT` is chosen.** Three candidates are computed from the pixels before any search:
- **`TOTSU`**: the lower 3-class multi-Otsu threshold over tissue pixels only, with glass (gray > 215) excluded.
- **`W128` and `W512`**: the same threshold over a click-centred square of 128 or 512 µm, truncated at the ROI edge.

The searched one is the eligible candidate whose `log(b / b_otsu)` has the largest SD over clicks. That is the one that
departs most unevenly from `b_otsu`, the adversarial pick, because the ladder already covers uniform shifts.

**`SFILL` is a diagnostic only.** It is the seed's own surround fill, the click-only floor. It is not searched: the
product will search ROIs like these, and a seed beside glass would give it a near-zero floor by construction.

## Metrics

The three co-primaries are all on `tm_score`, summed per ROI over its 6 clicks:
- **P@10**;
- **P@20**;
- **mitoses in the post-NMS pool** (up to 100 detections), per click. A later feature re-ranker can only promote what
  the matcher put in the pool.

Descriptive metrics:
- P@30;
- white-adjacent and other top-10 false positives, and M2 (the white-adjacent share of top-10 false positives);
- the worst click per ROI, zero-TP@10 clicks, and per-click SD;
- per-domain P@10 and P@20;
- template flags: SD, the share of template and kept-box pixels sitting at the floor, and mass weight <= 0.

## Margins and statistics

**Non-inferiority to `A3`.** An arm passes on a co-primary when the lower bound of the ROI-cluster bootstrap 95 % CI
(`N_BOOT` 10000, `BOOT_SEED` 20260917, the source's ROI order) clears the margin.
- **P@10 and P@20:** -1.0 pt. That is about a third of the clip's gain (3.30 / 2.62 pts).
- **Pool:** -0.5 mitoses per click, about a third of the +1.52 gain.
- **Holm families.** The exact sign-flip p goes into Holm within each family:
  - "loss vs `A3`": 5 arms x {P@10, P@20, pool};
  - "gain vs `A1`": 6 arms x {P@10, P@20, P@30}.
- All statistics functions are verbatim copies of the source notebook's, asserted.

## Decision rules, fixed now

1. **Plateau.** The plateau holds if `L050` and `L150` are both non-inferior to `A3` on all three co-primaries. Then
   **c = 1.0 stays**: it is the centre of a level range the gain does not depend on.
2. **The sweep cannot move the level.**
   - If `L025` is also non-inferior, report "a near-glass floor suffices". Moving the level needs held-out data, not an
     in-sample sweep.
   - If the plateau fails on one side, report it and stop. No post hoc level is chosen.
3. **Estimator.** If `ALT` is non-inferior on all three, the ROI-wide Otsu is not special. If not, the estimator matters
   and the ROI-wide Otsu stays.
4. **Red flags, reported beside the result and not used to decide:**
   - any Holm-significant loss vs `A3`;
   - a non-inferior arm losing more than 2 pts in one domain;
   - degenerate templates;
   - a zig-zag on the ladder, meaning two consecutive steps of opposite sign with both CIs clear of 0.
5. **Primaries that disagree.** If P@K and pool disagree on an arm, the disagreement is reported; neither is picked.

## Predictions, written before the run

- **P1. The plateau holds.** `b_med`, a different estimator at a different clip share, already tied `b_otsu`.
- **P2. `L025` is uncertain.** At about 0.008 OD it clips mainly glass and the palest pixels. The verdict notebook found
  that candidates with only 2-10 % white are already affected, so part of the gain may need more than glass removed.
- **P3. `L200` loses.** Its floor is about 0.9x the seed's kept-box mean at the median, so it clips into chromatin.
- **P4. White-adjacent top-10 false positives fall with c and saturate.**
  - P@10 should follow them.
  - If P@10 keeps rising after white-FP removal saturates, something other than the white surround is driving the gain.
  - This ladder replaces the image-side vs template-side split as the mechanism check.

## Gates (halt on failure)

1. The configuration matches `midog_utils.production`.
2. The search helpers and statistics functions are verbatim copies of the source notebook's def blocks.
3. The per-ROI `b_otsu` and `b_med` equal `floors.csv`, and the unclipped seed template equals the recorded SHA-1, on
   every click.
4. **Parity.** `A3` on all 294 clicks and `A1` on click 0 of every ROI reproduce the recorded pool exactly:
   - coordinates, float32 scores, od, buckets, threshold, n_detections and template SHA-1;
   - TP@10/20/30 and pool mitoses.
5. The clip identity and the blank fill equal `b` on every search.
6. The copied statistics reproduce the recorded `A3 - A1` rows.
7. The pool mitoses re-counted from the saved pools equal the per-run column.

## What this cannot answer

- **Stain and scanner shift** is the stress test, a separate experiment.
- **Held-out ROIs.** Every number is in-sample on the 49 ROIs the clip was found on.
- **Later rounds** of the interactive loop are not simulated.
- **Chromatin od** is not analysed.

Runtime: about 37 searches per ROI at about 2.5 s each, plus one floor pass. That is roughly 80-90 min. A smoke run on
3 ROIs (`TMFLOOR_SMOKE_ROIS=337.tiff,509.tiff,201.tiff`) comes first and writes `_smoke` copies.

In [1]:
import gc
import hashlib
import inspect
import json
import math
import os
import pickle
import sys
import time
from fractions import Fraction

import cv2
import numpy as np
import pandas as pd
from scipy import stats as sps
from skimage.filters import threshold_multiotsu

sys.path.insert(0, '../..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import invariants as inv
from midog_utils import production as prod
from midog_utils import template_match as tm
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 400)
pd.set_option('display.max_colwidth', 140)

NB_T0 = time.time()

# gate 1 -- the configuration, asserted against midog_utils.production (as in the source notebook)
REFERENCE_CONFIG = dict(CHANNEL='hematoxylin_od', TM_METHOD=cv2.TM_CCOEFF, PEAK_MIN_DISTANCE=7, DEEP_FLOOR_Z=-1.5, MAX_PEAKS=100, OD_WINDOW=51)
assert all(getattr(prod, k) == v for k, v in REFERENCE_CONFIG.items()), 'gate 1: midog_utils.production no longer matches the configuration the recorded run used'
assert ev.MIDOG_RADIUS_UM == 7.5 and (tm.BASE_SIZE, tm.PATCH_SIZE) == (51, 73)
assert hasattr(tm, 'blank_seed_square') and not hasattr(prod, 'SELF_HIT_RADIUS'), 'this is not post-D11 code'
assert prod.AXES == {'tm_score': 'score', 'chromatin_od': 'od'}

# constants the copied helpers read, with the source notebook's values (cell 1 of tm_border_floor_49roi_6seed.ipynb)
BLANK_SIZE = 51
HALF_BLANK = BLANK_SIZE // 2
OD_PAD = 25
MAX_SIDE = 2 * (tm.PATCH_SIZE // 2) + 1
EDGE_READ_PAD = tm.PATCH_SIZE // 2
RING_PX = 5
ARMS = ('mo_box', 'mo_pad51', 'mo_pad61', 'mo_ring5')   # build_mo_templates' own arm set; only mo_pad61 is used
PAD_ARMS = ('mo_pad51', 'mo_pad61')
AXIS_KEY = {'chromatin_od': 'od', 'tm_score': 'score'}
BUDGETS = (10, 20, 30)
TEST_K = 10
SELF_PX = 2
SEED_INDICES = (0, 1, 2, 3, 4, 5)
N_CLICKS_PER_ROI = len(SEED_INDICES)
N_BOOT = 10_000
BOOT_SEED = 20260917
SUBSET_DIRS = {'original_14': '../../images/extra_valid', 'testing_35': '../../images/extra_valid/testing_set'}
TPL_ARM = 'mo_pad61'
S_TPL = 61
PAD = (S_TPL - 1) // 2
FLOOR_CLASSES = 3
WHITE_WINDOW = 61
WHITE_GRAY = 215
WHITE_SHARE = 0.20
ALPHA = 0.05
assert AXIS_KEY == prod.AXES and OD_PAD == prod.OD_WINDOW // 2

# this notebook (fixed in the pre-registration, cell 0)
RANK_KEY = 'tm_score'          # the production ranker (D5); chromatin od is computed by the copied detect but not analysed
BAND_MODE = 'floored'          # the corrected benchmark harness of A1/A3
LADDER = {'L025': 0.25, 'L050': 0.5, 'A3': 1.0, 'L150': 1.5, 'L200': 2.0}   # floor = c x the ROI's lower 3-class multi-Otsu threshold
WINDOWS_UM = {'W128': 128.0, 'W512': 512.0}   # click-centred squares, truncated at the ROI edge
ALT_CANDIDATES = ('TOTSU', 'W128', 'W512')    # one is searched: the widest spread of log(b / b_otsu) over clicks, chosen before any search
NI_MARGIN = 0.01               # non-inferiority margin vs A3 on P@10 and P@20 (1.0 precision point)
NI_MARGIN_POOL = 0.5           # non-inferiority margin vs A3 on mitoses in the post-NMS pool, per click
DOMAIN_FLAG = 0.02             # a non-inferior arm losing more than 2 points in one domain is flagged
PARITY_SEEDS_A1 = (0,)         # A1 (no clip) is re-searched on these clicks of every ROI; A3 on all

SMOKE_ROIS = [s for s in os.environ.get('TMFLOOR_SMOKE_ROIS', '').split(',') if s] or None
SUFFIX = '' if SMOKE_ROIS is None else '_smoke'
SRC_NB = 'tm_border_floor_49roi_6seed.ipynb'
REC = '../../results/tm_border_floor_49roi_6seed'
FG_SRC = '../../results/precision_at_k_49roi_6seed_fillgeom_postD11'
DB_PATH = '../../databases/MIDOG++.json'
INPUTS = {'rec_per_run': f'{REC}_per_run.csv', 'rec_delta_stats': f'{REC}_delta_stats.csv', 'rec_floors': f'{REC}_floors.csv', 'rec_search_pickle': f'{REC}_search.pkl', 'fillgeom_per_run': f'{FG_SRC}_per_run.csv', 'midog_json': DB_PATH, 'source_notebook': SRC_NB}
PICKLE_SHA256 = '0f921c65acaa1164b8475d8c266b936123a0718a07a4d0f010162cca8cec7d69'
OUT_STEM = f'../../results/tm_floor_definition_49roi_6seed{SUFFIX}'
OUT = {name: f'{OUT_STEM}_{name}.csv' for name in ('per_run', 'top30', 'floors', 'contrasts', 'arm_summary', 'ladder', 'domain', 'safety', 'decisions', 'adversarial_log')}
OUT['pool'] = f'{OUT_STEM}_pool.csv.gz'
assert not set(map(os.path.abspath, OUT.values())) & set(map(os.path.abspath, INPUTS.values())), 'an output path would overwrite an input'
if SMOKE_ROIS is None:
    existing = [p for p in OUT.values() if os.path.exists(p)]
    assert not existing, f'the full run never overwrites its outputs; these already exist: {existing}'

# the adversarial log: each concern is registered here, before any search, and answered by the cell that tests it
ADV = {}


def adv(item, concern, test):
    """
        Register one adversarial concern and the test that answers it, before the test runs.

        item (str): a short key.
        concern (str): what could make the experiment fail or mislead.
        test (str): how this notebook tests it.

        Returns None.
    """
    ADV[item] = dict(item=item, concern=concern, test=test, result='', action='')


def adv_result(item, result, action):
    """
        Fill in one adversarial concern's result and the action taken.

        item (str): the key it was registered under.
        result (str): what the test found.
        action (str): what was done about it.

        Returns None.
    """
    assert item in ADV, f'{item} was never registered'
    ADV[item].update(result=result, action=action)


adv('code_path', 'a new search path could differ from the recorded one, so any floor effect would be partly a code effect', 'A3 re-searched on every click and A1 on click 0 of every ROI; pools, thresholds, template SHA-1s and TP@K must equal the recorded run exactly')
adv('floor_reproduces', 'the per-ROI b_otsu recomputed here could differ from the recorded floors (float32 rounding, channel drift)', 'exact equality with floors.csv on every ROI before any search')
adv('degenerate_template', 'at high floors the seed template can flatten (SD 0, mass weight <= 0), and a degenerate template would read as a level effect', 'template SD, share of template and kept-box pixels at the floor, and mass weight per click and arm; flagged clicks reported separately')
adv('alt_selection_bias', 'choosing the alternative estimator after seeing outcomes would bias it', 'the alternative is chosen from floor values alone (widest spread of log(b / b_otsu) over clicks), printed and fixed before the search cell runs')
adv('margin_choice', 'the non-inferiority margins are mine, so a pass could be an artefact of a generous margin', 'margins fixed in cell 0 as about a third of the clip gain; every CI is reported so a reader can apply another margin')
adv('mechanism', 'precision could keep rising with the floor for a reason other than the white surround', 'white-adjacent and other top-10 false positives and M2 per ladder level; precision is compared with where white-FP removal saturates')
adv('zigzag', 'a non-monotone ladder larger than its CIs is more likely a bug than biology', 'adjacent-level contrasts at K = 10; a sign reversal with both CIs excluding 0 is flagged and investigated')
adv('window_truncation', 'click-centred windows are truncated at the ROI edge, so their effective size varies', 'effective window area per click reported')
adv('selfmatch', 'a clipped blank could change the self-match picture', 'selfmatch_counts on every list; any top-10 self-match is reported')
adv('in_sample', 'every arm is scored on the 49 ROIs the clip was found on', 'the level cannot be moved by this sweep (cell 0); a lower level is reported, not adopted')

print('ladder:', LADDER)
print(f'smoke ROIs = {SMOKE_ROIS}')
print('outputs (all written at the end):', *OUT.values(), sep='\n  ')


ladder: {'L025': 0.25, 'L050': 0.5, 'A3': 1.0, 'L150': 1.5, 'L200': 2.0}
smoke ROIs = None
outputs (all written at the end):
  ../../results/tm_floor_definition_49roi_6seed_per_run.csv
  ../../results/tm_floor_definition_49roi_6seed_top30.csv
  ../../results/tm_floor_definition_49roi_6seed_floors.csv
  ../../results/tm_floor_definition_49roi_6seed_contrasts.csv
  ../../results/tm_floor_definition_49roi_6seed_arm_summary.csv
  ../../results/tm_floor_definition_49roi_6seed_ladder.csv
  ../../results/tm_floor_definition_49roi_6seed_domain.csv
  ../../results/tm_floor_definition_49roi_6seed_safety.csv
  ../../results/tm_floor_definition_49roi_6seed_decisions.csv
  ../../results/tm_floor_definition_49roi_6seed_adversarial_log.csv
  ../../results/tm_floor_definition_49roi_6seed_pool.csv.gz


## Inputs, hashed

In [2]:
def sha256_12(path):
    """
        The first 12 hex digits of a file's SHA-256, so the inputs this notebook read are on record.

        path (str): the file to hash.

        Returns str: 12 hex characters.
    """
    with open(path, 'rb') as fh:
        return hashlib.sha256(fh.read()).hexdigest()[:12]


INPUT_SHA = {name: sha256_12(p) for name, p in INPUTS.items()}
for name, p in INPUTS.items():
    print(f'  {INPUT_SHA[name]}  {p}')
assert INPUT_SHA['rec_search_pickle'] == PICKLE_SHA256[:12], 'the search pickle is not the recorded run'

REC_RUN = pd.read_csv(INPUTS['rec_per_run'], float_precision='round_trip')
REC_RUN = REC_RUN[REC_RUN['rank_key'] == RANK_KEY].reset_index(drop=True)
REC_FLOORS = pd.read_csv(INPUTS['rec_floors'], float_precision='round_trip').set_index('file_name')
with open(INPUTS['rec_search_pickle'], 'rb') as fh:
    REC_POOL = pickle.load(fh)['POOL']
REC_POOL = REC_POOL[REC_POOL['arm'].isin(['A1', 'A3'])]
REC_POOL_IX = {key: g.sort_values('rank').reset_index(drop=True) for key, g in REC_POOL.groupby(['file_name', 'seed_index', 'arm'])}
del REC_POOL
gc.collect()

ROI_ORDER_ALL = list(pd.unique(pd.read_csv(INPUTS['fillgeom_per_run'], usecols=['file_name'])['file_name']))
assert len(ROI_ORDER_ALL) == 49 and set(ROI_ORDER_ALL) == set(REC_RUN['file_name'])
ROI_ORDER = ROI_ORDER_ALL if SMOKE_ROIS is None else [f for f in ROI_ORDER_ALL if f in SMOKE_ROIS]
assert SMOKE_ROIS is None or sorted(ROI_ORDER) == sorted(SMOKE_ROIS), f'smoke ROIs not found: {SMOKE_ROIS}'
N_ROIS = len(ROI_ORDER)

# the clicks and their recorded refinement boxes (gate 2 of the source run reproduced them from the pixels; the audit re-checked)
CLK = REC_RUN[REC_RUN['arm'] == 'A0'][['subset', 'domain', 'file_name', 'seed_index', 'seed_ann_id', 'click_cx', 'click_cy', 'tcx', 'tcy', 'bs', 'mpp', 'match_radius_px', 'fill_value', 'm_keep', 'tpl_sha1']].rename(columns={'fill_value': 'fill_unclipped', 'm_keep': 'm_keep_unclipped', 'tpl_sha1': 'tpl_sha1_unclipped'})
CLK = CLK[CLK['file_name'].isin(ROI_ORDER)].sort_values(['file_name', 'seed_index']).reset_index(drop=True)
assert len(CLK) == 6 * N_ROIS and CLK.groupby('file_name')['seed_index'].apply(lambda v: tuple(sorted(v)) == SEED_INDICES).all()
REC_IX = REC_RUN.set_index(['file_name', 'seed_index', 'arm']).sort_index()
assert (REC_IX.xs('A1', level='arm')['tpl_sha1'] == REC_IX.xs('A0', level='arm')['tpl_sha1']).all(), 'A0 and A1 should share the unclipped template'
DOMAIN_OF = CLK.drop_duplicates('file_name').set_index('file_name')['domain']
DOMAINS = list(pd.unique(CLK['domain']))

images, annotations = ds.load_annotations(DB_PATH)
print(f'\n{N_ROIS} ROIs, {len(CLK)} clicks; recorded pools for A1 and A3 loaded for the parity gates')


  ef7940000fec  ../../results/tm_border_floor_49roi_6seed_per_run.csv
  e87350f523e7  ../../results/tm_border_floor_49roi_6seed_delta_stats.csv
  a8e83aa2692b  ../../results/tm_border_floor_49roi_6seed_floors.csv
  0f921c65acaa  ../../results/tm_border_floor_49roi_6seed_search.pkl
  8668d2697420  ../../results/precision_at_k_49roi_6seed_fillgeom_postD11_per_run.csv
  8c87ab4276eb  ../../databases/MIDOG++.json
  0e8338c36e60  tm_border_floor_49roi_6seed.ipynb



49 ROIs, 294 clicks; recorded pools for A1 and A3 loaded for the parity gates


## Search helpers, verbatim from the source notebook (asserted)

In [3]:
def fill_outside(T, keep, fill=None):
    """
        A copy of the template with every pixel outside the keep mask set to one fill value.

        T (np.ndarray): the W x W float32 window.
        keep (np.ndarray): the W x W boolean keep mask.
        fill (np.float32 or None): the fill value; None fills with the mean of the outside pixels on this window.

        Returns tuple[np.ndarray, np.float32, int]: the filled template, the fill value, and the number of filled pixels.
    """
    outside = ~keep
    assert outside.any(), 'the keep region covers the whole window, so there is nothing to fill'
    value = np.float32(T[outside].astype(np.float64).mean()) if fill is None else np.float32(fill)
    F = np.array(T, dtype=np.float32, copy=True)
    F[outside] = value
    return np.ascontiguousarray(F), value, int(outside.sum())


def template_sha1(T):
    """
        SHA-1 of a template's float32 bytes, the digest the recorded run stores as tpl_sha1.

        T (np.ndarray): the float32 template.

        Returns str: the hex digest.
    """
    return hashlib.sha1(np.ascontiguousarray(T, dtype=np.float32).tobytes()).hexdigest()


def padded(channel, pad):
    """
        A channel replicate-padded on every side.

        channel (np.ndarray): the channel to pad.
        pad (int): pixels added on every side.

        Returns np.ndarray: the padded channel.
    """
    return cv2.copyMakeBorder(channel, pad, pad, pad, pad, borderType=cv2.BORDER_REPLICATE)


def read_window(hem, hem_edge, tcx, tcy, S):
    """
        An S x S window of the channel centred on (tcx, tcy), cut from a replicate-padded copy when it crosses the ROI edge.

        hem (np.ndarray): this ROI's hematoxylin_od channel.
        hem_edge (np.ndarray): the same channel replicate-padded by EDGE_READ_PAD on every side.
        tcx (int): window centre x, full-image pixels, already rounded.
        tcy (int): window centre y, full-image pixels, already rounded.
        S (int): the odd window side.

        Returns tuple[np.ndarray, bool]: a float32 copy of the window, and True when read_padded_patch refused it (roi_edge_read).
    """
    half = S // 2
    y0, x0 = tcy + EDGE_READ_PAD - half, tcx + EDGE_READ_PAD - half
    assert y0 >= 0 and x0 >= 0 and y0 + S <= hem_edge.shape[0] and x0 + S <= hem_edge.shape[1], f'the {S} px window at ({tcx}, {tcy}) runs past the {EDGE_READ_PAD} px replicate pad'
    edge = hem_edge[y0:y0 + S, x0:x0 + S]
    patch = tm.read_padded_patch(hem, tcx, tcy, S)
    if patch is not None:
        assert np.array_equal(patch, edge), 'the replicate-padded read disagrees with read_padded_patch inside the ROI'
        return np.array(patch, dtype=np.float32, copy=True), False
    return np.array(edge, dtype=np.float32, copy=True), True


def arm_side(arm, bs):
    """
        An arm's nominal template side for a box of side bs.

        arm (str): one of ARMS.
        bs (int): the box side.

        Returns int: bs, 51, 61 or bs + 2 * RING_PX.
    """
    return {'mo_box': bs, 'mo_pad51': 51, 'mo_pad61': 61, 'mo_ring5': bs + 2 * RING_PX}[arm]


def build_mo_templates(hem, hem_edge, tcx, tcy, bs):
    """
        The four arms' templates for one accepted click, each cut from its own fresh window at the box centre.

        hem (np.ndarray): this ROI's hematoxylin_od channel.
        hem_edge (np.ndarray): the same channel replicate-padded by EDGE_READ_PAD.
        tcx (int): box centre x, rounded once.
        tcy (int): box centre y, rounded once.
        bs (int): the odd box side.

        Returns tuple[dict, np.ndarray]: arm -> dict(T, raw, keep, S, S_nominal, fill, n_filled, n_kept, m_keep, mass_weight, roi_edge_read, box_ge_S), and the raw bs x bs box.
    """
    box = tm.read_padded_patch(hem, tcx, tcy, bs)
    assert box is not None, f'the {bs} px box at ({tcx}, {tcy}) is not readable'
    box = np.array(box, dtype=np.float32, copy=True)
    arms = {}
    for arm in ARMS:
        S_nominal = arm_side(arm, bs)
        box_ge_S = arm in PAD_ARMS and bs >= S_nominal
        S = bs if box_ge_S else S_nominal
        assert S % 2 == 1 and S <= MAX_SIDE, f'{arm}: template side {S} is even or above {MAX_SIDE}'
        raw, edge_read = read_window(hem, hem_edge, tcx, tcy, S)
        o = (S - bs) // 2
        keep = np.zeros((S, S), bool)
        keep[o:o + bs, o:o + bs] = True
        m_keep = float(raw[keep].astype(np.float64).mean())
        if S == bs:
            T, f, n_filled = np.ascontiguousarray(np.array(raw, copy=True)), np.float32(np.nan), 0
        else:
            T, f, n_filled = fill_outside(np.array(raw, copy=True), keep)
        arms[arm] = dict(T=T, raw=raw, keep=keep, S=S, S_nominal=S_nominal, fill=f, n_filled=n_filled, n_kept=bs * bs, m_keep=m_keep, mass_weight=m_keep - float(f) if n_filled else np.nan, roi_edge_read=edge_read, box_ge_S=box_ge_S)
    assert tuple(arms) == ARMS, 'the arm set changed'
    return arms, box


def rank_by(det, rank_key):
    """
        One ranking axis applied to a detection list, in run_production_pipeline's own sort order.

        det (pd.DataFrame): the unranked detections from search_once.
        rank_key (str): "chromatin_od" or "tm_score".

        Returns pd.DataFrame: the detections ranked best-first, with rank reassigned.
    """
    out = det.sort_values(AXIS_KEY[rank_key], ascending=False, na_position='last', kind='mergesort').reset_index(drop=True)
    return out.assign(rank=np.arange(len(out)))


def score_top_k(detections, gt_eval, match_radius):
    """
        Match a ranked detection list to ground truth and count true positives in each top-K.

        detections (pd.DataFrame): ranked detections, best first.
        gt_eval (pd.DataFrame): ground truth with the click's own annotation removed.
        match_radius (float): match radius in pixels.

        Returns tuple[pd.DataFrame, dict]: the detections with match buckets added, and K -> true positives in the top K.
    """
    det_out, _ = ev.bucket_detections(detections, gt_eval, match_radius)
    hit = (det_out['bucket'] == ev.HUMAN_CORRECT_LABEL).to_numpy()
    return det_out, {k: int(hit[:k].sum()) for k in BUDGETS}


def selfmatch_mask(frame, cx, cy, tcx, tcy):
    """
        Which detections are a self-match: not a true positive, and within SELF_PX (Chebyshev) of the template centre or of the click.

        frame (pd.DataFrame): matched detections with cx, cy and bucket.
        cx (float or np.ndarray): click x, unrounded.
        cy (float or np.ndarray): click y, unrounded.
        tcx (int or np.ndarray): template centre x.
        tcy (int or np.ndarray): template centre y.

        Returns tuple[np.ndarray, np.ndarray, np.ndarray]: the self-match mask, and which of its rows are near the click and near the centre.
    """
    x, y = frame['cx'].to_numpy(dtype=float), frame['cy'].to_numpy(dtype=float)
    near_click = np.maximum(np.abs(x - np.asarray(cx, dtype=float)), np.abs(y - np.asarray(cy, dtype=float))) <= SELF_PX
    near_centre = np.maximum(np.abs(x - np.asarray(tcx, dtype=float)), np.abs(y - np.asarray(tcy, dtype=float))) <= SELF_PX
    not_tp = (frame['bucket'] != ev.HUMAN_CORRECT_LABEL).to_numpy()
    return (near_click | near_centre) & not_tp, near_click, near_centre


def selfmatch_counts(out, cx, cy, tcx, tcy):
    """
        Self-match counts in one ranked, matched list.

        out (pd.DataFrame): one axis's ranked, matched detections.
        cx (float): click x.
        cy (float): click y.
        tcx (int): template centre x.
        tcy (int): template centre y.

        Returns dict: self-matches in the top 10, the top 30 and the whole list, and the top-30 split by where they sit.
    """
    m, near_click, near_centre = selfmatch_mask(out, cx, cy, tcx, tcy)
    return dict(n_selfmatch_top10=int(m[:TEST_K].sum()), n_selfmatch_top30=int(m[:30].sum()), n_selfmatch_all=int(m.sum()), n_selfmatch_top30_near_click=int((m & near_click)[:30].sum()), n_selfmatch_top30_near_centre_only=int((m & near_centre & ~near_click)[:30].sum()))


def roi_floors(hem):
    """
        The two per-ROI floors over the whole unclipped channel: the lower 3-class multi-Otsu threshold and the median.

        hem (np.ndarray): this ROI's hematoxylin_od channel.

        Returns dict: b_otsu, b_otsu_upper (the second threshold, for reference) and b_med, as float32-representable floats.
    """
    th = threshold_multiotsu(hem, classes=FLOOR_CLASSES)
    return dict(b_otsu=float(np.float32(th[0])), b_otsu_upper=float(np.float32(th[1])), b_med=float(np.float32(np.median(hem))))


def clip_floor(hem, b):
    """
        The floor-clipped channel, with the clip-identity gate (gate 4) asserted.

        hem (np.ndarray): this ROI's float32 hematoxylin_od channel.
        b (float): the floor.

        Returns np.ndarray: np.maximum(hem, b) as float32.
    """
    b32 = np.float32(b)
    hem_c = np.maximum(hem, b32)
    assert hem_c.dtype == np.float32 and hem_c.shape == hem.shape
    above = hem >= b32
    assert (hem_c >= hem).all() and np.array_equal(hem_c[above], hem[above]), 'gate 4: the clip changed a pixel at or above the floor, or lowered one'
    assert hem.min() < b32 and hem_c.min() == b32, f'gate 4: hem.min() {hem.min()} is not below the floor {b32}, so the blank fill would not be b'
    return hem_c


def band_mask(shape, px):
    """
        The border band: True within px pixels of the ROI edge.

        shape (tuple): (h, w).
        px (int): the band width.

        Returns np.ndarray: boolean (h, w).
    """
    m = np.zeros(shape, bool)
    m[:px] = m[-px:] = True
    m[:, :px] = m[:, -px:] = True
    return m


def assert_band_identity(band, px):
    """
        Gate 5, part 1: the band is exactly the pixels with edge_px < px.

        band (np.ndarray): the boolean band mask.
        px (int): the band width.

        Returns None.
    """
    h, w = band.shape
    ys = np.arange(h, dtype=np.int32)[:, None]
    xs = np.arange(w, dtype=np.int32)[None, :]
    dist = np.minimum(np.minimum(ys, h - 1 - ys), np.minimum(xs, w - 1 - xs))
    assert np.array_equal(band, dist < px), 'gate 5: the band is not exactly edge_px < PAD'


def fused_map(search_padded, template, pad, shape):
    """
        One template's TM_CCOEFF response over the ROI, cropped to ROI coordinates, as search_once computes it.

        search_padded (np.ndarray): the D11-blanked search channel, replicate-padded by pad.
        template (np.ndarray): the W x W float32 template, W = 2 * pad + 1.
        pad (int): the search channel's replicate pad.
        shape (tuple): the unpadded channel's (h, w).

        Returns tuple[np.ndarray, np.ndarray]: the fused map and its reachability mask, both (h, w).
    """
    assert template.shape == (2 * pad + 1, 2 * pad + 1), f'template {template.shape} does not match pad {pad}'
    h, w = shape
    fused_p, _, valid_p = tm.fused_response(search_padded, [template], False, method=prod.TM_METHOD)
    fused = fused_p[pad:pad + h, pad:pad + w]
    valid = valid_p[pad:pad + h, pad:pad + w]
    assert valid.all(), f'the {pad} px pad left part of the ROI unreachable'
    return fused, valid


def detect(fused, valid, band, band_mode, od_padded, od_pad, nms_radius):
    """
        search_once after the correlation, with the border band handled one of three ways.

        fused (np.ndarray): the fused map from fused_map.
        valid (np.ndarray): its reachability mask.
        band (np.ndarray): the border band mask.
        band_mode (str): 'production' (band searched), 'invalid' (the prompt's spec: valid False in the band), or 'floored' (valid False and the band's scores floored before extract_peaks).
        od_padded (np.ndarray): the UNCLIPPED, UNBLANKED hematoxylin_od channel padded by od_pad.
        od_pad (int): the od channel's replicate pad.
        nms_radius (float): this ROI's NMS radius in pixels.

        Returns tuple[pd.DataFrame, dict, np.ndarray]: unranked detections (cx, cy, score, od), the search's counts and statistics, and the pre-NMS peak centres.
    """
    h, w = fused.shape
    if band_mode == 'production':
        f, v = fused, valid
    else:
        assert band_mode in ('invalid', 'floored'), band_mode
        v = valid & ~band
        f = fused if band_mode == 'invalid' else np.where(band, np.float32(tm._FLOOR), fused)
    med, mad = tm.robust_stats(f, v)
    threshold = med + prod.DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(f, v, prod.PEAK_MIN_DISTANCE, threshold, prod.MAX_PEAKS)
    peaks = centers.copy()
    keep = nms_by_distance(centers, scores, nms_radius)
    centers, scores = centers[keep], scores[keep]
    det = pd.DataFrame({'rank': np.arange(len(centers)), 'cx': centers[:, 0], 'cy': centers[:, 1], 'score': scores})
    assert od_padded.shape[:2] == (h + 2 * od_pad, w + 2 * od_pad), f'the od channel is not padded by {od_pad}'
    shifted = det.assign(cx=det['cx'] + od_pad, cy=det['cy'] + od_pad)
    det = det.assign(od=cm.score_detections(shifted, od_padded, window=prod.OD_WINDOW)['od'].to_numpy())
    info = dict(n_peaks=len(peaks), max_peaks_binding=bool(len(peaks) == prod.MAX_PEAKS), n_detections=len(det), threshold=float(threshold), median=float(med), mad=float(mad))
    return det, info, peaks


def white_integral(rgb):
    """
        The summed-area table of the white mask on a replicate-padded gray copy, so any 61 px white share is O(1).

        rgb (np.ndarray): the ROI's uint8 RGB image.

        Returns np.ndarray: cv2.integral of (gray > WHITE_GRAY), padded by WHITE_WINDOW // 2.
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    white = (padded(gray, WHITE_WINDOW // 2) > WHITE_GRAY).astype(np.uint8)
    return cv2.integral(white)


def white_frac(integral, cx, cy):
    """
        The share of white pixels in the 61 px window centred on each detection, on the unclipped RGB.

        integral (np.ndarray): white_integral of the ROI.
        cx (np.ndarray): detection x, integer-valued ROI pixels.
        cy (np.ndarray): detection y, integer-valued ROI pixels.

        Returns np.ndarray: white share in [0, 1].
    """
    x, y = np.asarray(cx).astype(np.int64), np.asarray(cy).astype(np.int64)
    assert np.array_equal(x, np.asarray(cx)) and np.array_equal(y, np.asarray(cy)), 'detection centres are not integer pixels'
    s = WHITE_WINDOW
    total = integral[y + s, x + s] - integral[y, x + s] - integral[y + s, x] + integral[y, x]
    return total / float(s * s)


def def_source(cell_src, name):
    """
        One top-level function's source as written in a notebook cell, from its def line to the next top-level line.

        cell_src (str): the cell's source.
        name (str): the function name.

        Returns str: the def block, trailing blank lines removed.
    """
    lines = cell_src.split('\n')
    start = next(i for i, ln in enumerate(lines) if ln.startswith(f'def {name}('))
    end = next((i for i in range(start + 1, len(lines)) if lines[i] and not lines[i][0].isspace()), len(lines))
    return '\n'.join(lines[start:end]).rstrip()


SRC_CELLS = [''.join(c['source']) for c in json.load(open(SRC_NB))['cells']]
COPIED_HELPERS = {6: ('fill_outside', 'template_sha1', 'padded', 'read_window', 'arm_side', 'build_mo_templates', 'rank_by', 'score_top_k', 'selfmatch_mask', 'selfmatch_counts'), 8: ('roi_floors', 'clip_floor', 'band_mask', 'assert_band_identity', 'fused_map', 'detect', 'white_integral', 'white_frac')}
for cell, names in COPIED_HELPERS.items():
    for name in names:
        assert inspect.getsource(globals()[name]).rstrip() == def_source(SRC_CELLS[cell], name), f'{name} is not a verbatim copy of cell {cell} of {SRC_NB}'
print(f"verbatim: {sum(len(v) for v in COPIED_HELPERS.values())} search helpers equal their def blocks in cells 6 and 8 of {SRC_NB}")


verbatim: 18 search helpers equal their def blocks in cells 6 and 8 of tm_border_floor_49roi_6seed.ipynb


## Statistics, verbatim, and checked against the recorded `A3 - A1` rows

In [4]:
def sign_flip_null_counts(abs_deltas):
    """
        Exact null distribution of sum(+/-|d_i|) by convolution over integer offsets.

        abs_deltas (array-like): nonzero absolute integer deltas, one per ROI.

        Returns np.ndarray: int64 counts of each sum -T..T (index = sum + T, T = sum |d_i|), totalling 2 ** n.
    """
    d = [int(v) for v in abs_deltas]
    assert all(v > 0 for v in d) and len(d) <= 62, 'nonzero integer deltas only, and 2 ** n must fit int64'
    total = sum(d)
    counts = np.zeros(2 * total + 1, dtype=np.int64)
    counts[total] = 1
    for v in d:
        shifted = np.zeros_like(counts)
        shifted[v:] += counts[:len(counts) - v]
        shifted[:len(counts) - v] += counts[v:]
        counts = shifted
    return counts


def exact_sign_flip_p(deltas):
    """
        Two-sided exact sign-flip p on integer per-ROI deltas, by dynamic-programming convolution.

        deltas (array-like): one integer delta per ROI; zeros carry no sign.

        Returns tuple[float, int]: the p-value and the number of nonzero deltas.
    """
    d = np.asarray(deltas)
    assert np.all(d == np.round(d)), 'the convolution needs integer deltas'
    d = d.astype(np.int64)
    nz = np.abs(d[d != 0])
    if len(nz) == 0:
        return 1.0, 0
    counts = sign_flip_null_counts(nz)
    total = int(nz.sum())
    observed = abs(int(d.sum()))
    sums = np.arange(-total, total + 1)
    return float(Fraction(int(counts[np.abs(sums) >= observed].sum()), 2 ** len(nz))), len(nz)


def holm_adjust(pvalues):
    """
        Holm step-down adjusted p-values, valid under any dependence between the tests.

        pvalues (array-like): raw p-values of one family of tests.

        Returns np.ndarray: adjusted p-values, in the input order.
    """
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p, kind='stable')
    adjusted = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]).clip(max=1.0)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    return out


def boot_index(n_roi, stream=0):
    """
        The cluster-bootstrap resample of ROI indices, seeded exactly as every earlier run seeds it.

        n_roi (int): the number of ROIs being resampled.
        stream (int): 0 for the reported CI; 1.. for the re-seeding stability check.

        Returns np.ndarray: (N_BOOT, n_roi) resampled ROI indices.
    """
    return np.random.default_rng([BOOT_SEED, stream]).integers(n_roi, size=(N_BOOT, n_roi))


def boot_ci_exact(int_values, idx, denominator):
    """
        Percentile 95% CI of a pooled ratio over resampled ROIs, from exact integer numerators.

        int_values (np.ndarray): one integer numerator per ROI.
        idx (np.ndarray): (n_boot, n_roi) resampled ROI indices.
        denominator (int): the pooled denominator per resample.

        Returns tuple[float, float, float, float]: CI low, CI high, and the share of resamples at or above 0 and at or below 0.
    """
    sums = np.asarray(int_values, dtype=np.int64)[idx].sum(axis=1)
    lo, hi = np.percentile(sums / denominator, [2.5, 97.5])
    return float(lo), float(hi), float((sums >= 0).mean()), float((sums <= 0).mean())


def pair_stats(d, k, idx):
    """
        Every statistic for one contrast at one budget, from its per-ROI click-summed integer deltas.

        d (np.ndarray): one integer delta per ROI, in ROI order.
        k (int): the budget K the deltas count true positives in.
        idx (np.ndarray): (N_BOOT, len(d)) bootstrap ROI indices.

        Returns dict: one delta_stats row without its Holm columns.
    """
    d = np.asarray(d, dtype=np.int64)
    n_roi = len(d)
    denominator = N_CLICKS_PER_ROI * n_roi * k
    pooled = d.sum() / denominator
    wins, losses, ties = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
    exact_p, n_nonzero = exact_sign_flip_p(d)
    ci_low, ci_high, share_ge_0, share_le_0 = boot_ci_exact(d, idx, denominator)
    x = d / (N_CLICKS_PER_ROI * k)
    half = sps.t.ppf(0.975, n_roi - 1) * x.std(ddof=1) / math.sqrt(n_roi) if n_roi > 1 else np.nan
    t_ci_low, t_ci_high = float(x.mean() - half), float(x.mean() + half)
    ci_excludes_0 = bool(ci_low > 0 or ci_high < 0)
    n_same_direction = wins if pooled > 0 else (losses if pooled < 0 else 0)
    majority = bool(n_same_direction > n_roi / 2)
    return dict(n_roi=n_roi, n_clicks=N_CLICKS_PER_ROI * n_roi, pooled_delta_precision=pooled, wins=wins, losses=losses, ties=ties, n_nonzero=n_nonzero, exact_p=exact_p, min_attainable_p=2.0 ** (1 - n_nonzero) if n_nonzero else 1.0, ci_low=ci_low, ci_high=ci_high, boot_share_ge_0=share_ge_0, boot_share_le_0=share_le_0, t_ci_low=t_ci_low, t_ci_high=t_ci_high, ci_excludes_0=ci_excludes_0, t_ci_excludes_0=bool(t_ci_low > 0 or t_ci_high < 0), n_same_direction=n_same_direction, majority_of_all_rois=majority, d5_bar_met=bool(ci_excludes_0 and majority))


def roi_sum(frame, axis, arm, col):
    """
        One column summed over each ROI's clicks for one arm and axis, in ROI_ORDER.

        frame (pd.DataFrame): a per-run table.
        axis (str): the rank_key.
        arm (str): the arm.
        col (str): the column to sum.

        Returns np.ndarray: int64, one value per ROI.
    """
    g = frame[(frame['rank_key'] == axis) & (frame['arm'] == arm)].groupby('file_name')[col].sum()
    return g.reindex(ROI_ORDER).to_numpy(dtype=np.int64)


def share_diff_boot(num_a, den_a, num_b, den_b, idx):
    """
        The paired difference of two ratio-of-sums shares, with a ROI-cluster bootstrap 95% CI.

        num_a (np.ndarray): per-ROI numerators of arm a.
        den_a (np.ndarray): per-ROI denominators of arm a.
        num_b (np.ndarray): per-ROI numerators of arm b.
        den_b (np.ndarray): per-ROI denominators of arm b.
        idx (np.ndarray): (N_BOOT, n_roi) resampled ROI indices.

        Returns dict: the two shares, their difference, its CI and whether the CI lies below 0.
    """
    sa, sb = num_a.sum() / den_a.sum(), num_b.sum() / den_b.sum()
    da, db = den_a[idx].sum(axis=1), den_b[idx].sum(axis=1)
    ok = (da > 0) & (db > 0)
    diff = num_a[idx].sum(axis=1)[ok] / da[ok] - num_b[idx].sum(axis=1)[ok] / db[ok]
    lo, hi = np.percentile(diff, [2.5, 97.5])
    return dict(share_a=sa, share_b=sb, diff=sa - sb, ci_low=float(lo), ci_high=float(hi), ci_below_0=bool(hi < 0), ci_above_0=bool(lo > 0), n_boot_used=int(ok.sum()))


def count_delta(d, idx):
    """
        A per-ROI integer count delta: its total, the exact two-sided sign-flip p, and a ROI-cluster bootstrap 95% CI of the total.

        d (np.ndarray): one integer delta per ROI.
        idx (np.ndarray): (N_BOOT, n_roi) resampled ROI indices.

        Returns dict: total, per click, wins/losses/ties, exact p, CI of the total.
    """
    d = np.asarray(d, dtype=np.int64)
    p, n_nz = exact_sign_flip_p(d)
    lo, hi = np.percentile(d[idx].sum(axis=1), [2.5, 97.5])
    return dict(total=int(d.sum()), per_click=d.sum() / (N_CLICKS_PER_ROI * len(d)), wins=int((d > 0).sum()), losses=int((d < 0).sum()), ties=int((d == 0).sum()), exact_p=p, n_nonzero=n_nz, ci_low=float(lo), ci_high=float(hi))


COPIED_STATS = {10: ('sign_flip_null_counts', 'exact_sign_flip_p', 'holm_adjust', 'boot_index', 'boot_ci_exact', 'pair_stats'), 21: ('roi_sum', 'share_diff_boot', 'count_delta')}
for cell, names in COPIED_STATS.items():
    for name in names:
        assert inspect.getsource(globals()[name]).rstrip() == def_source(SRC_CELLS[cell], name), f'{name} is not a verbatim copy of cell {cell} of {SRC_NB}'

# the copied statistics reproduce the recorded A3 - A1 rows (tm_score, K = 10/20/30) from the recorded per-run CSV
rec_delta = pd.read_csv(INPUTS['rec_delta_stats'], float_precision='round_trip')
if SMOKE_ROIS is None:
    for k in BUDGETS:
        d = roi_sum(REC_RUN, RANK_KEY, 'A3', f'tp_at_{k}') - roi_sum(REC_RUN, RANK_KEY, 'A1', f'tp_at_{k}')
        st = pair_stats(d, k, boot_index(N_ROIS))
        r = rec_delta[(rec_delta['rank_key'] == RANK_KEY) & (rec_delta['pair'] == 'A3 - A1') & (rec_delta['K'] == k)].iloc[0]
        assert all(np.isclose(st[c], r[c], rtol=1e-12, atol=0) for c in ('pooled_delta_precision', 'ci_low', 'ci_high', 'exact_p', 't_ci_low', 't_ci_high')), f'stats do not reproduce the recorded A3 - A1 at K = {k}'
    print('stats: verbatim (9 functions) and they reproduce the recorded A3 - A1 rows at K = 10/20/30')
else:
    print('stats: verbatim (9 functions); the recorded-row check runs on the full set only')


stats: verbatim (9 functions) and they reproduce the recorded A3 - A1 rows at K = 10/20/30


## Every candidate floor, before any search, and the choice of `ALT`

These are computed from the pixels alone. For each candidate floor the table gives:
- its ratio to `b_otsu`;
- its ratio to the seed's kept-box mean (how close it comes to the seed's chromatin);
- the share of the ROI it clips;
- the shares of the seed's 61 px window and kept box that lie below it.

In [5]:
def window_floor(hem, cx, cy, side):
    """
        The lower 3-class multi-Otsu threshold over a square window centred on a click, truncated at the ROI edge.

        hem (np.ndarray): the unclipped channel.
        cx (float): click x.
        cy (float): click y.
        side (int): the odd window side in px.

        Returns tuple[float, int]: the float32-representable floor and the window's pixel count after truncation.
    """
    h, w = hem.shape
    half = side // 2
    ix, iy = int(round(cx)), int(round(cy))
    win = hem[max(0, iy - half):min(h, iy + half + 1), max(0, ix - half):min(w, ix + half + 1)]
    return float(np.float32(threshold_multiotsu(win, classes=FLOOR_CLASSES)[0])), int(win.size)


# every candidate floor, from the pixels alone, before any search: the ladder, the tissue-only Otsu, two click-centred windows, and the seed's own surround fill
floor_rows = []
t0 = time.time()
for f in ROI_ORDER:
    rows = CLK[CLK['file_name'] == f]
    path = f"{SUBSET_DIRS[rows['subset'].iat[0]]}/{f}"
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    assert mpp == rows['mpp'].iat[0], f'{f}: mpp changed'
    hem = ch.to_channel(rgb, 'hematoxylin_od')
    white = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) > WHITE_GRAY
    del rgb
    fl = roi_floors(hem)
    assert fl['b_otsu'] == REC_FLOORS.loc[f, 'b_otsu'] and fl['b_med'] == REC_FLOORS.loc[f, 'b_med'], f'{f}: the ROI floors do not reproduce floors.csv'
    b_otsu = fl['b_otsu']
    b_tissue = float(np.float32(threshold_multiotsu(hem[~white], classes=FLOOR_CLASSES)[0]))
    hem_edge = padded(hem, EDGE_READ_PAD)
    for r in rows.itertuples():
        a = build_mo_templates(hem, hem_edge, int(r.tcx), int(r.tcy), int(r.bs))[0][TPL_ARM]
        assert template_sha1(a['T']) == r.tpl_sha1_unclipped, f'{f} s{r.seed_index}: the unclipped template is not the recorded one'
        cand = {arm: float(np.float32(c * b_otsu)) for arm, c in LADDER.items()}
        cand['TOTSU'] = b_tissue
        for arm, um in WINDOWS_UM.items():
            side = int(round(um / mpp)) | 1
            cand[arm], n_px = window_floor(hem, r.click_cx, r.click_cy, side)
            cand[f'{arm}_area_share'] = n_px / side ** 2
        cand['SFILL'] = float(np.float32(r.fill_unclipped))
        for arm, b in cand.items():
            if arm.endswith('_area_share'):
                continue
            b32 = np.float32(b)
            floor_rows.append(dict(file_name=f, domain=r.domain, seed_index=int(r.seed_index), arm=arm, b=b, b_over_otsu=b / b_otsu, b_over_mkeep=b / r.m_keep_unclipped, roi_clip_share=float((hem < b32).mean()), tpl_window_share_below=float((a['raw'] < b32).mean()), kept_share_below=float((a['raw'][a['keep']] < b32).mean()), window_area_share=cand.get(f'{arm}_area_share', np.nan), b_gt_hem_min=bool(b32 > hem.min())))
    del hem, hem_edge, white
    gc.collect()
FLOORS = pd.DataFrame(floor_rows)
print(f'floors for {FLOORS["arm"].nunique()} candidates on {len(CLK)} clicks in {(time.time() - t0) / 60:.1f} min; per-ROI b_otsu and b_med equal floors.csv on all {N_ROIS} ROIs')

summ = FLOORS.groupby('arm', sort=False).agg(b_median=('b', 'median'), ratio_otsu_median=('b_over_otsu', 'median'), ratio_otsu_p05=('b_over_otsu', lambda v: v.quantile(0.05)), ratio_otsu_p95=('b_over_otsu', lambda v: v.quantile(0.95)), log_ratio_sd=('b_over_otsu', lambda v: float(np.log(v).std())), b_over_mkeep_median=('b_over_mkeep', 'median'), b_over_mkeep_max=('b_over_mkeep', 'max'), roi_clip_share_median=('roi_clip_share', 'median'), kept_share_below_median=('kept_share_below', 'median'), kept_share_below_max=('kept_share_below', 'max'), tpl_window_share_below_median=('tpl_window_share_below', 'median'), window_area_share_min=('window_area_share', 'min'), all_above_hem_min=('b_gt_hem_min', 'all'))
print('\nevery candidate floor (ratio to the ROI b_otsu; ratio to the seed kept-box mean; shares below the floor):')
print(summ.to_string(float_format=lambda v: f'{v:.3f}'))

# the searched alternative: the eligible candidate whose level departs most unevenly from b_otsu (the ladder already covers uniform shifts)
elig = summ.loc[list(ALT_CANDIDATES)]
elig = elig[elig['all_above_hem_min']]
ALT = str(elig['log_ratio_sd'].idxmax())
SEARCH_ARMS = list(LADDER) + [ALT]
print(f"\nALT = {ALT}: the widest spread of log(b / b_otsu) over clicks among {list(elig.index)} ({elig['log_ratio_sd'].round(3).to_dict()}); fixed before any search")
print('searched arms:', SEARCH_ARMS, '+ A1 on clicks', PARITY_SEEDS_A1)
adv_result('alt_selection_bias', f"ALT = {ALT}, chosen by log-ratio SD {elig['log_ratio_sd'].round(3).to_dict()} before the search cell", 'fixed; SFILL kept as a diagnostic only (cell 0)')
ws = FLOORS[FLOORS['arm'].isin(WINDOWS_UM)].groupby('arm')['window_area_share']
adv_result('window_truncation', '; '.join(f'{a}: median {v.median():.2f}, min {v.min():.2f} of the nominal area' for a, v in ws), 'reported; truncation is part of what the window estimator does on a cropped ROI')
FLOOR_IX = FLOORS.set_index(['file_name', 'seed_index', 'arm'])['b']


floors for 9 candidates on 294 clicks in 10.0 min; per-ROI b_otsu and b_med equal floors.csv on all 49 ROIs

every candidate floor (ratio to the ROI b_otsu; ratio to the seed kept-box mean; shares below the floor):
       b_median  ratio_otsu_median  ratio_otsu_p05  ratio_otsu_p95  log_ratio_sd  b_over_mkeep_median  b_over_mkeep_max  roi_clip_share_median  kept_share_below_median  kept_share_below_max  tpl_window_share_below_median  window_area_share_min  all_above_hem_min
arm                                                                                                                                                                                                                                                                   
L025      0.007              0.250           0.250           0.250         0.000                0.110             0.282                  0.135                    0.047                 0.456                          0.107                    NaN               T

## The search: 37 searches per ROI, with parity gates inline

In [6]:
def search_arm(channel, channel_edge, b, cx, cy, tcx, tcy, bs, band, od_padded, nms_radius):
    """
        One arm's search on one click: the mo_pad61 template from the given channel, the D11 blank, the correlation, and detection on the corrected harness.

        channel (np.ndarray): the search channel, clipped or not.
        channel_edge (np.ndarray): the same channel replicate-padded by EDGE_READ_PAD, for the template read.
        b (float or None): the floor the channel was clipped at; None when unclipped.
        cx (float): click x, the blank's centre.
        cy (float): click y.
        tcx (int): box centre x.
        tcy (int): box centre y.
        bs (int): the box side.
        band (np.ndarray): the border band mask.
        od_padded (np.ndarray): the unclipped channel padded by OD_PAD.
        nms_radius (float): the NMS radius in px.

        Returns tuple[pd.DataFrame, dict]: unranked detections, and the search's statistics with template diagnostics.
    """
    a = build_mo_templates(channel, channel_edge, tcx, tcy, bs)[0][TPL_ARM]
    assert a['S'] == S_TPL and not a['box_ge_S'], f'the {TPL_ARM} template is not {S_TPL} px'
    search, n_blanked = tm.blank_seed_square(channel, cx, cy, BLANK_SIZE)
    assert n_blanked == BLANK_SIZE ** 2, f'gate 4: {n_blanked} px blanked'
    blank_fill = search[int(round(cy)), int(round(cx))]
    assert blank_fill == channel.min() and (b is None or blank_fill == np.float32(b)), f'gate 4: the blank fill {blank_fill} is not the floor {b}'
    fused, valid = fused_map(padded(search, PAD), a['T'], PAD, channel.shape)
    det, info, _ = detect(fused, valid, band, BAND_MODE, od_padded, OD_PAD, nms_radius)
    T = a['T'].astype(np.float64)
    b32 = None if b is None else np.float32(b)
    tpl = dict(tpl_sha1=template_sha1(a['T']), fill_value=float(a['fill']), m_keep=a['m_keep'], mass_weight=a['mass_weight'], tpl_sd=float(T.std()), tpl_share_at_floor=np.nan if b is None else float((a['T'] == b32).mean()), kept_share_at_floor=np.nan if b is None else float((a['raw'][a['keep']] == b32).mean()))
    tpl['degenerate'] = bool(tpl['tpl_sd'] < 1e-6 or not tpl['mass_weight'] > 0)
    return det, dict(info, **tpl)


def tp_rows(det, gt_eval, match_radius, wint, r, arm, b, info, t_s):
    """
        Rank one arm's detections by tm_score, match them, and summarise the list.

        det (pd.DataFrame): unranked detections.
        gt_eval (pd.DataFrame): ground truth without the click's own annotation.
        match_radius (float): the match radius in px.
        wint (np.ndarray): the ROI's white integral.
        r (tuple): the click row.
        arm (str): the arm.
        b (float or None): the floor.
        info (dict): search_arm's statistics.
        t_s (float): seconds this arm's search took.

        Returns tuple[dict, pd.DataFrame]: the per-run row and the ranked, matched pool with white fractions.
    """
    ranked = rank_by(det, RANK_KEY)
    out, tp = score_top_k(ranked, gt_eval, match_radius)
    out = out.assign(white_frac_61=white_frac(wint, out['cx'], out['cy']) if len(out) else np.zeros(0))
    is_tp = (out['bucket'] == ev.HUMAN_CORRECT_LABEL).to_numpy()
    white = (out['white_frac_61'] > WHITE_SHARE).to_numpy()
    row = dict(subset=r.subset, domain=r.domain, file_name=r.file_name, seed_index=int(r.seed_index), seed_ann_id=int(r.seed_ann_id), arm=arm, rank_key=RANK_KEY, floor=np.nan if b is None else float(b), **{f'tp_at_{k}': tp[k] for k in BUDGETS}, n_mit_pool=int(is_tp.sum()), n_fp_top10=int((~is_tp[:10]).sum()), n_white_fp_top10=int((~is_tp & white)[:10].sum()), n_tp_top30=int(is_tp[:30].sum()), n_white_tp_top30=int((is_tp & white)[:30].sum()), t_search_s=t_s, **info, **selfmatch_counts(out, r.click_cx, r.click_cy, r.tcx, r.tcy))
    return row, out


def parity(out, info, r, arm):
    """
        Gate 3: a re-searched reference arm equals the recorded run (pool coordinates, scores, od, buckets, threshold, template, TP@K).

        out (pd.DataFrame): the ranked, matched pool.
        info (dict): search_arm's statistics.
        r (tuple): the click row.
        arm (str): 'A1' or 'A3'.

        Returns None.
    """
    tag = f'{r.file_name} s{r.seed_index} {arm}'
    rec = REC_IX.loc[(r.file_name, int(r.seed_index), arm)]
    pool = REC_POOL_IX[(r.file_name, int(r.seed_index), arm)]
    assert info['tpl_sha1'] == rec['tpl_sha1'], f'{tag}: template differs from the recorded one'
    assert info['threshold'] == rec['threshold'] and info['n_detections'] == rec['n_detections'], f'{tag}: threshold or n_detections differs'
    assert len(out) == len(pool) and np.array_equal(out[['cx', 'cy']].to_numpy(), pool[['cx', 'cy']].to_numpy()) and np.array_equal(out['score'].to_numpy(np.float32), pool['score'].to_numpy(np.float32)) and np.isclose(out['od'].to_numpy(), pool['od'].to_numpy(), rtol=1e-12, atol=0).all() and (out['bucket'].to_numpy() == pool['bucket'].to_numpy()).all(), f'{tag}: the pool differs from the recorded one'
    hit = (out['bucket'] == ev.HUMAN_CORRECT_LABEL).to_numpy()
    assert all(int(hit[:k].sum()) == int(rec[f'tp_at_{k}']) for k in BUDGETS) and int(hit.sum()) == int(rec['n_mit_pool']), f'{tag}: TP@K or pool mitoses differ'


run_rows, top_frames, pool_frames = [], [], []
n_parity = {'A1': 0, 'A3': 0}
t_loop = time.time()
for i, f in enumerate(ROI_ORDER):
    rows = CLK[CLK['file_name'] == f]
    path = f"{SUBSET_DIRS[rows['subset'].iat[0]]}/{f}"
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    hem = ch.to_channel(rgb, 'hematoxylin_od')
    wint = white_integral(rgb)
    del rgb
    od_padded = padded(hem, OD_PAD)
    band = band_mask(hem.shape, PAD)
    assert_band_identity(band, PAD)
    gt = ds.image_annotations(annotations, f)
    nms_radius = match_radius = ev.radius_px(mpp)
    inv.check_nms_radius(nms_radius, mpp, label='tm_floor_definition')
    assert all(np.isclose(match_radius, rows['match_radius_px'], rtol=1e-12, atol=0))
    gts = {int(r.seed_index): gt[gt['ann_id'] != r.seed_ann_id].reset_index(drop=True) for r in rows.itertuples()}
    plan = [('A1', None, [r for r in rows.itertuples() if int(r.seed_index) in PARITY_SEEDS_A1])]
    for arm in SEARCH_ARMS:
        by_b = {}
        for r in rows.itertuples():
            by_b.setdefault(float(FLOOR_IX[(f, int(r.seed_index), arm)]), []).append(r)
        plan += [(arm, b, rs) for b, rs in by_b.items()]
    for arm, b, rs in plan:
        channel = hem if b is None else clip_floor(hem, b)
        channel_edge = padded(channel, EDGE_READ_PAD)
        for r in rs:
            t0 = time.perf_counter()
            det, info = search_arm(channel, channel_edge, b, float(r.click_cx), float(r.click_cy), int(r.tcx), int(r.tcy), int(r.bs), band, od_padded, nms_radius)
            t_s = time.perf_counter() - t0
            row, out = tp_rows(det, gts[int(r.seed_index)], match_radius, wint, r, arm, b, info, t_s)
            if arm in n_parity:
                parity(out, info, r, arm)
                n_parity[arm] += 1
            if arm == 'A1':
                continue
            run_rows.append(row)
            top_frames.append(out.head(30)[['rank', 'cx', 'cy', 'score', 'od', 'bucket', 'matched_ann_id', 'white_frac_61']].assign(file_name=f, seed_index=int(r.seed_index), arm=arm))
            pool_frames.append(out[['rank', 'cx', 'cy', 'score', 'od', 'bucket', 'matched_ann_id', 'white_frac_61']].assign(file_name=f, seed_index=int(r.seed_index), arm=arm))
        del channel, channel_edge
    del hem, od_padded, band, wint
    gc.collect()
    print(f'[{i + 1}/{N_ROIS} {f}] {(time.time() - t_loop) / 60:.1f} min', flush=True)

NEW = pd.DataFrame(run_rows)
TOP = pd.concat(top_frames, ignore_index=True)
POOL = pd.concat(pool_frames, ignore_index=True)
assert len(NEW) == len(SEARCH_ARMS) * len(CLK) and not NEW.duplicated(['file_name', 'seed_index', 'arm']).any()
assert n_parity == {'A1': N_ROIS * len(PARITY_SEEDS_A1), 'A3': len(CLK)}, n_parity
print(f"\ngate 3 held: A3 re-searched on all {n_parity['A3']} clicks and A1 on {n_parity['A1']} reproduce the recorded pools, thresholds, templates and TP@K exactly")
print(f"loop {(time.time() - t_loop) / 60:.1f} min; median {NEW['t_search_s'].median():.2f} s per arm-search")
adv_result('code_path', f"A3 equal on {n_parity['A3']} clicks, A1 on {n_parity['A1']}", 'the new arms share this code path')


[1/49 300.tiff] 1.6 min


[2/49 301.tiff] 3.1 min


[3/49 322.tiff] 4.2 min


[4/49 337.tiff] 5.3 min


[5/49 339.tiff] 7.4 min


...

[46/49 364.tiff] 61.6 min


[47/49 400.tiff] 63.0 min


[48/49 401.tiff] 66.2 min


[49/49 404.tiff] 67.5 min



gate 3 held: A3 re-searched on all 294 clicks and A1 on 49 reproduce the recorded pools, thresholds, templates and TP@K exactly
loop 67.5 min; median 1.79 s per arm-search


## Per-run table, self-matches and template flags

In [7]:
# the per-run table: the searched arms, plus the recorded A1 (no clip) on every click, which the parity gate tied to this code path
rec_a1 = REC_RUN[(REC_RUN['arm'] == 'A1') & REC_RUN['file_name'].isin(ROI_ORDER)].copy()
rec_a1['floor'] = np.nan
keep_cols = [c for c in NEW.columns if c in rec_a1.columns]
PER_RUN = pd.concat([rec_a1[keep_cols], NEW[keep_cols + [c for c in NEW.columns if c not in keep_cols]]], ignore_index=True)
ARM_ORDER = ['A1'] + SEARCH_ARMS
LEVEL = {'A1': 0.0, **LADDER}
assert set(PER_RUN['arm']) == set(ARM_ORDER) and len(PER_RUN) == len(ARM_ORDER) * len(CLK)

# gate 8 and the template flags
sm = PER_RUN.groupby('arm', sort=False)[['n_selfmatch_top10', 'n_selfmatch_top30']].sum().reindex(ARM_ORDER)
print('self-matches per arm (top 10 / top 30):', sm.to_dict('index'))
adv_result('selfmatch', f"top-10 self-matches: {int(sm['n_selfmatch_top10'].sum())}; top-30: {int(sm['n_selfmatch_top30'].sum())}", 'none to drop' if int(sm['n_selfmatch_top10'].sum()) == 0 else 'reported per arm; contrasts are not re-run without them')
flags = NEW.groupby('arm', sort=False).agg(degenerate=('degenerate', 'sum'), tpl_share_at_floor_median=('tpl_share_at_floor', 'median'), tpl_share_at_floor_max=('tpl_share_at_floor', 'max'), kept_share_at_floor_median=('kept_share_at_floor', 'median'), kept_share_at_floor_max=('kept_share_at_floor', 'max'), mass_weight_median=('mass_weight', 'median'), tpl_sd_min=('tpl_sd', 'min'), max_peaks_binding=('max_peaks_binding', 'mean'), n_detections_min=('n_detections', 'min')).reindex(SEARCH_ARMS)
print('\ntemplate flags per arm (share of template and kept-box pixels sitting at the floor; degenerate = SD 0 or mass weight <= 0):')
print(flags.to_string(float_format=lambda v: f'{v:.3f}'))
DEGENERATE = NEW.loc[NEW['degenerate'], ['file_name', 'seed_index', 'arm', 'tpl_sd', 'mass_weight', 'kept_share_at_floor', 'tp_at_10']]
if len(DEGENERATE):
    print('\ndegenerate templates:')
    print(DEGENERATE.to_string(index=False))
adv_result('degenerate_template', f"degenerate per arm: {flags['degenerate'].astype(int).to_dict()}; kept-box share at the floor, max per arm: {flags['kept_share_at_floor_max'].round(2).to_dict()}", 'flagged clicks listed with their TP@10; read those arms with them in mind' if len(DEGENERATE) else 'none')


self-matches per arm (top 10 / top 30): {'A1': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'L025': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'L050': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'A3': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'L150': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'L200': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}, 'W128': {'n_selfmatch_top10': 0, 'n_selfmatch_top30': 0}}

template flags per arm (share of template and kept-box pixels sitting at the floor; degenerate = SD 0 or mass weight <= 0):
      degenerate  tpl_share_at_floor_median  tpl_share_at_floor_max  kept_share_at_floor_median  kept_share_at_floor_max  mass_weight_median  tpl_sd_min  max_peaks_binding  n_detections_min
arm                                                                                                                                                                                          
L025           3                      0.012         

## Contrasts: each arm vs `A3` (non-inferiority) and vs `A1` (does it keep the clip's gain)

In [8]:
IDX = boot_index(N_ROIS)
COMPARE = [a for a in SEARCH_ARMS if a != 'A3']
con_rows = []
for arm in COMPARE + ['A3']:
    for ref in (['A3', 'A1'] if arm != 'A3' else ['A1']):
        for k in BUDGETS:
            d = roi_sum(PER_RUN, RANK_KEY, arm, f'tp_at_{k}') - roi_sum(PER_RUN, RANK_KEY, ref, f'tp_at_{k}')
            con_rows.append(dict(arm=arm, ref=ref, metric=f'P@{k}', **pair_stats(d, k, IDX)))
        d = roi_sum(PER_RUN, RANK_KEY, arm, 'n_mit_pool') - roi_sum(PER_RUN, RANK_KEY, ref, 'n_mit_pool')
        cd = count_delta(d, IDX)
        n_clk = N_CLICKS_PER_ROI * N_ROIS
        con_rows.append(dict(arm=arm, ref=ref, metric='pool', n_roi=N_ROIS, n_clicks=n_clk, pooled_delta_precision=cd['per_click'], wins=cd['wins'], losses=cd['losses'], ties=cd['ties'], n_nonzero=cd['n_nonzero'], exact_p=cd['exact_p'], ci_low=cd['ci_low'] / n_clk, ci_high=cd['ci_high'] / n_clk))
CON = pd.DataFrame(con_rows)
# non-inferiority vs A3 on the three co-primaries; Holm families: loss vs A3 (P@10, P@20, pool) and gain vs A1 (P@10, P@20, P@30)
vs3 = (CON['ref'] == 'A3')
CON.loc[vs3, 'margin'] = np.where(CON.loc[vs3, 'metric'] == 'pool', -NI_MARGIN_POOL, -NI_MARGIN)
CON.loc[vs3, 'non_inferior'] = CON.loc[vs3, 'ci_low'] > CON.loc[vs3, 'margin']
fam3 = vs3 & CON['metric'].isin(['P@10', 'P@20', 'pool'])
CON.loc[fam3, 'holm_p'] = holm_adjust(CON.loc[fam3, 'exact_p'].to_numpy())
fam1 = (CON['ref'] == 'A1') & CON['metric'].isin(['P@10', 'P@20', 'P@30'])
CON.loc[fam1, 'holm_p'] = holm_adjust(CON.loc[fam1, 'exact_p'].to_numpy())
CON['significant_loss'] = (CON['holm_p'] < ALPHA) & (CON['pooled_delta_precision'] < 0)
CON['significant_gain'] = (CON['holm_p'] < ALPHA) & (CON['pooled_delta_precision'] > 0)
pts = CON.copy()
is_p = pts['metric'] != 'pool'
for c in ('pooled_delta_precision', 'ci_low', 'ci_high', 't_ci_low', 't_ci_high', 'margin'):
    if c in pts:
        pts.loc[is_p, c] = 100 * pts.loc[is_p, c]
show = ['arm', 'ref', 'metric', 'pooled_delta_precision', 'ci_low', 'ci_high', 't_ci_low', 't_ci_high', 'wins', 'losses', 'ties', 'exact_p', 'holm_p', 'margin', 'non_inferior', 'd5_bar_met', 'significant_loss', 'significant_gain']
print('every contrast (P@K in points; pool in mitoses per click; margin is the non-inferiority bound vs A3):')
print(pts[show].to_string(index=False, float_format=lambda v: f'{v:.3g}'))


every contrast (P@K in points; pool in mitoses per click; margin is the non-inferiority bound vs A3):
 arm ref metric  pooled_delta_precision  ci_low  ci_high  t_ci_low  t_ci_high  wins  losses  ties  exact_p   holm_p  margin non_inferior d5_bar_met  significant_loss  significant_gain
L025  A3   P@10                   -2.45   -3.47     -1.5     -3.47      -1.43     8      32     9 8.22e-06 9.86e-05      -1        False       True              True             False
L025  A3   P@20                   -2.11   -2.91    -1.38      -2.9      -1.31     8      37     4 2.57e-07 3.35e-06      -1        False       True              True             False
L025  A3   P@30                   -1.81   -2.54    -1.15     -2.54      -1.09     7      39     3 1.01e-06      NaN      -1        False       True             False             False
L025  A3   pool                   -1.21    -1.6    -0.85       NaN        NaN     3      43     3 7.56e-10 1.13e-08    -0.5        False        NaN              T

## The ladder as a dose: precision against white-adjacent false-positive removal, and adjacent steps

In [9]:
# the ladder as a dose: precision, pool, and white-adjacent vs other top-10 false positives at each floor level
lad_rows = []
for arm in ['A1'] + list(LADDER) + [ALT]:
    g = PER_RUN[PER_RUN['arm'] == arm]
    n = len(g)
    m2 = share_diff_boot(roi_sum(PER_RUN, RANK_KEY, arm, 'n_white_fp_top10'), roi_sum(PER_RUN, RANK_KEY, arm, 'n_fp_top10'), roi_sum(PER_RUN, RANK_KEY, 'A1', 'n_white_fp_top10'), roi_sum(PER_RUN, RANK_KEY, 'A1', 'n_fp_top10'), IDX)
    lad_rows.append(dict(arm=arm, c=LEVEL.get(arm, np.nan), **{f'P@{k}': g[f'tp_at_{k}'].sum() / (k * n) for k in BUDGETS}, pool_mit_per_click=g['n_mit_pool'].mean(), white_fp10=int(g['n_white_fp_top10'].sum()), nonwhite_fp10=int((g['n_fp_top10'] - g['n_white_fp_top10']).sum()), m2_share=m2['share_a'], m2_diff_vs_A1=m2['diff'], m2_ci_low=m2['ci_low'], m2_ci_high=m2['ci_high'], white_tp30=int(g['n_white_tp_top30'].sum()), roi_clip_share_median=FLOORS[FLOORS['arm'] == arm]['roi_clip_share'].median() if arm != 'A1' else 0.0, kept_share_below_median=FLOORS[FLOORS['arm'] == arm]['kept_share_below'].median() if arm != 'A1' else 0.0))
LADDER_TAB = pd.DataFrame(lad_rows)
print(f'the ladder (c = floor / b_otsu; A1 is c = 0, no clip), and {ALT} (c undefined: a different estimator) last:')
print(LADDER_TAB.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

# adjacent-level contrasts at K = 10: a sign reversal with both CIs excluding 0 would be a zig-zag
adj_rows = []
steps = ['A1'] + list(LADDER)
for lo_arm, hi_arm in zip(steps[:-1], steps[1:]):
    for k in (10, 20):
        d = roi_sum(PER_RUN, RANK_KEY, hi_arm, f'tp_at_{k}') - roi_sum(PER_RUN, RANK_KEY, lo_arm, f'tp_at_{k}')
        st = pair_stats(d, k, IDX)
        adj_rows.append(dict(step=f'{hi_arm} - {lo_arm}', K=k, delta_pts=100 * st['pooled_delta_precision'], ci_low_pts=100 * st['ci_low'], ci_high_pts=100 * st['ci_high'], wlt=f"{st['wins']}/{st['losses']}/{st['ties']}"))
ADJ = pd.DataFrame(adj_rows)
print('\nadjacent ladder steps:')
print(ADJ.to_string(index=False, float_format=lambda v: f'{v:.2f}'))
sig = ADJ.assign(sgn=np.where(ADJ['ci_low_pts'] > 0, 1, np.where(ADJ['ci_high_pts'] < 0, -1, 0)))
ZIGZAG = {k: bool(any(a * b < 0 for a, b in zip(g['sgn'].to_numpy()[:-1], g['sgn'].to_numpy()[1:]))) for k, g in sig.groupby('K')}
print('zig-zag (two consecutive steps of opposite sign, both CIs clear of 0):', ZIGZAG)
adv_result('zigzag', f'{ZIGZAG}', 'investigate before reading' if any(ZIGZAG.values()) else 'none')


the ladder (c = floor / b_otsu; A1 is c = 0, no clip), and W128 (c undefined: a different estimator) last:
 arm      c   P@10   P@20   P@30  pool_mit_per_click  white_fp10  nonwhite_fp10  m2_share  m2_diff_vs_A1  m2_ci_low  m2_ci_high  white_tp30  roi_clip_share_median  kept_share_below_median
  A1 0.0000 0.5459 0.4723 0.4186             24.6837         316           1019    0.2367         0.0000     0.0000      0.0000         360                 0.0000                   0.0000
L025 0.2500 0.5544 0.4774 0.4235             24.9932         300           1010    0.2290        -0.0077    -0.0149     -0.0011         360                 0.1346                   0.0467
L050 0.5000 0.5650 0.4867 0.4307             25.4864         277           1002    0.2166        -0.0201    -0.0315     -0.0095         354                 0.2780                   0.1153
  A3 1.0000 0.5789 0.4985 0.4416             26.2075         240            998    0.1939        -0.0428    -0.0602     -0.0270         354  


adjacent ladder steps:
       step  K  delta_pts  ci_low_pts  ci_high_pts      wlt
  L025 - A1 10       0.85        0.41         1.36  17/2/30
  L025 - A1 20       0.51        0.24         0.82  20/3/26
L050 - L025 10       1.05        0.65         1.50  21/1/27
L050 - L025 20       0.94        0.60         1.31  30/4/15
  A3 - L050 10       1.39        0.58         2.28 25/12/12
  A3 - L050 20       1.17        0.61         1.77 28/11/10
  L150 - A3 10       0.27       -0.61         1.09 23/15/11
  L150 - A3 20       0.34       -0.37         1.11 20/18/11
L200 - L150 10      -0.27       -0.95         0.41 12/18/19
L200 - L150 20      -0.15       -0.71         0.39 18/18/13
zig-zag (two consecutive steps of opposite sign, both CIs clear of 0): {10: False, 20: False}


## The product view per arm, and per domain

In [10]:
# product-defect metrics per arm: the worst click per ROI, zero-TP@10 clicks, per-click spread, clicks losing 2+ vs A3 and vs A1
w10 = PER_RUN.pivot_table(index=['file_name', 'seed_index'], columns='arm', values='tp_at_10')
prod_rows = []
for arm in ARM_ORDER:
    worst = w10[arm].groupby('file_name').min()
    prod_rows.append(dict(arm=arm, c=LEVEL.get(arm, np.nan), P10=w10[arm].mean() / 10, mean_worst_click_tp10=worst.mean(), rois_worst_click_0=int((worst == 0).sum()), clicks_tp10_0=int((w10[arm] == 0).sum()), mean_click_sd_tp10=w10[arm].groupby('file_name').std().mean(), clicks_lose2_vs_A3=int(((w10[arm] - w10['A3']) <= -2).sum()), clicks_lose2_vs_A1=int(((w10[arm] - w10['A1']) <= -2).sum()), clicks_better_vs_A1=int((w10[arm] > w10['A1']).sum()), clicks_worse_vs_A1=int((w10[arm] < w10['A1']).sum())))
ARM_SUMMARY = pd.DataFrame(prod_rows).merge(LADDER_TAB[['arm', 'pool_mit_per_click', 'white_fp10', 'nonwhite_fp10', 'm2_share']], on='arm', how='left')
print('per arm, the product view (tm_score, K = 10):')
print(ARM_SUMMARY.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

# per domain, descriptive: P@10 and P@20 per arm, and each arm minus A3; a non-inferior arm losing more than 2 points in a domain is flagged
dom_rows = []
for dom in DOMAINS:
    g = PER_RUN[PER_RUN['domain'] == dom]
    n_clk = g[g['arm'] == 'A3'].shape[0]
    for k in (10, 20):
        row = dict(domain=dom, K=k)
        for arm in ARM_ORDER:
            row[arm] = 100 * g.loc[g['arm'] == arm, f'tp_at_{k}'].sum() / (k * n_clk)
        dom_rows.append(row)
DOMAIN_TAB = pd.DataFrame(dom_rows)
for arm in COMPARE + ['A1']:
    DOMAIN_TAB[f'{arm}-A3'] = DOMAIN_TAB[arm] - DOMAIN_TAB['A3']
print('\nper domain, P@K in points, and each arm minus A3:')
print(DOMAIN_TAB.to_string(index=False, float_format=lambda v: f'{v:.1f}'))


per arm, the product view (tm_score, K = 10):
 arm     c   P10  mean_worst_click_tp10  rois_worst_click_0  clicks_tp10_0  mean_click_sd_tp10  clicks_lose2_vs_A3  clicks_lose2_vs_A1  clicks_better_vs_A1  clicks_worse_vs_A1  pool_mit_per_click  white_fp10  nonwhite_fp10  m2_share
  A1 0.000 0.546                  4.163                   4             13               1.010                  19                   0                    0                   0              24.684         316           1019     0.237
L025 0.250 0.554                  4.306                   3             12               0.982                   8                   0                   27                   4              24.993         300           1010     0.229
L050 0.500 0.565                  4.388                   3             10               0.959                   5                   0                   58                   6              25.486         277           1002     0.217
  A3 1.000 0.579      

## Safety: what happens when the floor reaches into the seed's kept box

In [11]:
# does clipping into the seed's kept box hurt? per click and clipped arm, the share of the kept box at the floor against TP@10 minus A1
sf = NEW[['file_name', 'seed_index', 'domain', 'arm', 'kept_share_at_floor', 'floor', 'tp_at_10', 'tp_at_20', 'n_mit_pool', 'degenerate']].merge(PER_RUN[PER_RUN['arm'] == 'A1'][['file_name', 'seed_index', 'tp_at_10', 'tp_at_20', 'n_mit_pool']], on=['file_name', 'seed_index'], suffixes=('', '_A1'))
sf = sf.merge(CLK[['file_name', 'seed_index', 'm_keep_unclipped']], on=['file_name', 'seed_index'])
sf['b_over_mkeep'] = sf['floor'] / sf['m_keep_unclipped']
sf['d10'] = sf['tp_at_10'] - sf['tp_at_10_A1']
sf['d20'] = sf['tp_at_20'] - sf['tp_at_20_A1']
sf['dpool'] = sf['n_mit_pool'] - sf['n_mit_pool_A1']
KEPT_BINS = (-0.001, 0.05, 0.15, 0.30, 0.50, 1.0)
sf['kept_bin'] = pd.cut(sf['kept_share_at_floor'], KEPT_BINS)
SAFETY = sf.groupby(['arm', 'kept_bin'], observed=True, sort=False).agg(clicks=('d10', 'size'), d10_mean=('d10', 'mean'), d20_mean=('d20', 'mean'), dpool_mean=('dpool', 'mean'), b_over_mkeep_median=('b_over_mkeep', 'median')).reset_index()
SAFETY = SAFETY.assign(_o=SAFETY['arm'].map(SEARCH_ARMS.index)).sort_values(['_o', 'kept_bin']).drop(columns='_o').reset_index(drop=True)
print('per arm and share of the seed kept box clipped to the floor: mean change vs A1 per click (TP@10, TP@20, pool mitoses):')
print(SAFETY.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
within = []
for arm, g in sf.groupby('arm', sort=False):
    rho, p = sps.spearmanr(g['kept_share_at_floor'], g['d10']) if g['kept_share_at_floor'].nunique() > 1 else (np.nan, np.nan)
    within.append(dict(arm=arm, spearman_kept_vs_d10=rho, p=p))
print('\nwithin-arm Spearman, kept share at floor vs TP@10 change:', pd.DataFrame(within).round(3).to_dict('records'))
SAFETY_CLICKS = sf


per arm and share of the seed kept box clipped to the floor: mean change vs A1 per click (TP@10, TP@20, pool mitoses):
 arm       kept_bin  clicks  d10_mean  d20_mean  dpool_mean  b_over_mkeep_median
L025 (-0.001, 0.05]     153     0.065     0.085       0.242                0.111
L025   (0.05, 0.15]     107     0.112     0.150       0.364                0.108
L025    (0.15, 0.3]      28     0.036     0.071       0.500                0.113
L025     (0.3, 0.5]       6     0.333    -0.167       0.167                0.126
L050 (-0.001, 0.05]      78     0.167     0.321       0.846                0.207
L050   (0.05, 0.15]      95     0.200     0.347       0.789                0.221
L050    (0.15, 0.3]      97     0.186     0.216       0.773                0.221
L050     (0.3, 0.5]      23     0.217     0.261       0.826                0.269
L050     (0.5, 1.0]       1     1.000     0.000       1.000                0.306
  A3 (-0.001, 0.05]       3     0.333     0.667       1.000            

## Hand checks

In [12]:
# hand re-derivation of one contrast, without the copied functions: L050 - A3 at K = 10
a = PER_RUN[PER_RUN['arm'] == 'L050'].set_index(['file_name', 'seed_index'])['tp_at_10']
b = PER_RUN[PER_RUN['arm'] == 'A3'].set_index(['file_name', 'seed_index'])['tp_at_10']
by_hand = (a - b.reindex(a.index)).sum() / (10 * len(a))
row = CON[(CON['arm'] == 'L050') & (CON['ref'] == 'A3') & (CON['metric'] == 'P@10')].iloc[0]
assert abs(by_hand - row['pooled_delta_precision']) < 1e-15, (by_hand, row['pooled_delta_precision'])
print(f"hand check: L050 - A3 at K = 10 = {100 * by_hand:+.3f} pts from a plain groupby, equal to pair_stats")
# and the pool mitoses re-count from the saved pool equals the per-run column
pc = POOL.assign(tp=POOL['bucket'] == ev.HUMAN_CORRECT_LABEL).groupby(['file_name', 'seed_index', 'arm'])['tp'].sum()
assert (pc == NEW.set_index(['file_name', 'seed_index', 'arm'])['n_mit_pool'].reindex(pc.index)).all()
print('pool mitoses re-counted from the saved pools equal n_mit_pool on every searched list')


hand check: L050 - A3 at K = 10 = -1.395 pts from a plain groupby, equal to pair_stats


pool mitoses re-counted from the saved pools equal n_mit_pool on every searched list


## Decisions, as pre-registered, and outputs

In [13]:
def ni(arm, metric):
    """
        Whether one arm is non-inferior to A3 on one co-primary.

        arm (str): the arm.
        metric (str): 'P@10', 'P@20' or 'pool'.

        Returns bool: True when the CI's lower bound clears the margin.
    """
    return bool(CON[(CON['arm'] == arm) & (CON['ref'] == 'A3') & (CON['metric'] == metric)].iloc[0]['non_inferior'])


def ni_all(arm):
    """
        Whether one arm is non-inferior to A3 on all three co-primaries.

        arm (str): the arm.

        Returns bool: True when P@10, P@20 and pool all pass.
    """
    return all(ni(arm, m) for m in ('P@10', 'P@20', 'pool'))


PLATEAU = ni_all('L050') and ni_all('L150')
NEAR_GLASS = ni_all('L025')
ALT_OK = ni_all(ALT)
UPPER_EDGE = ni_all('L200')
LOSSES = CON[(CON['ref'] == 'A3') & CON['significant_loss']][['arm', 'metric']].to_dict('records')
dom_flags = []
for arm in COMPARE:
    if ni_all(arm):
        bad = DOMAIN_TAB[DOMAIN_TAB[f'{arm}-A3'] < -100 * DOMAIN_FLAG]
        dom_flags += [f"{arm} {r.domain} K={r.K} {r[f'{arm}-A3']:+.1f}" for _, r in bad.iterrows()]
if PLATEAU:
    LEVEL_RULE = 'plateau holds: keep c = 1.0 (b_otsu), the centre of a level range the gain does not depend on'
else:
    LEVEL_RULE = 'plateau fails on ' + ', '.join(a for a in ('L050', 'L150') if not ni_all(a)) + ': report and stop; no post hoc level'
DECISIONS = pd.DataFrame([
    dict(item='plateau (L050 and L150 non-inferior to A3 on P@10, P@20 and pool)', result=PLATEAU, detail=LEVEL_RULE),
    dict(item='near-glass floor suffices (L025 non-inferior on all three)', result=NEAR_GLASS, detail='reported only; the level is not moved in-sample'),
    dict(item=f'alternative estimator {ALT} non-inferior on all three', result=ALT_OK, detail='the ROI-wide Otsu is not special' if ALT_OK else 'the estimator matters: keep the ROI-wide Otsu'),
    dict(item='upper edge: L200 non-inferior on all three', result=UPPER_EDGE, detail='how far above b_otsu the floor can drift'),
    dict(item='any significant loss vs A3 (Holm, P@10 / P@20 / pool)', result=bool(LOSSES), detail=str(LOSSES)),
    dict(item='domain flags (non-inferior arm losing > 2 pts in a domain)', result=bool(dom_flags), detail='; '.join(dom_flags)),
    dict(item='zig-zag on the ladder', result=any(ZIGZAG.values()), detail=str(ZIGZAG)),
    dict(item='degenerate templates', result=bool(len(DEGENERATE)), detail=str(flags['degenerate'].astype(int).to_dict())),
])
print(DECISIONS.to_string(index=False))
print('\nper arm and co-primary, non-inferior to A3?')
print(CON[CON['ref'] == 'A3'].pivot_table(index='arm', columns='metric', values='non_inferior', aggfunc='first').reindex(COMPARE)[['P@10', 'P@20', 'pool']].to_string())
adv_result('margin_choice', f'margins {100 * NI_MARGIN:.1f} pt and {NI_MARGIN_POOL} mitoses per click; every CI is in contrasts.csv', 'fixed in cell 0')
lad = LADDER_TAB.set_index('arm')
adv_result('mechanism', 'white FP10 by level: ' + ', '.join(f"c={lad.loc[a, 'c']:.2f}: {int(lad.loc[a, 'white_fp10'])}" for a in lad.index) + '; P@10: ' + ', '.join(f"{lad.loc[a, 'P@10']:.3f}" for a in lad.index), 'read in the last cell')
adv_result('in_sample', 'all numbers in-sample on the 49 ROIs', 'no level change from this sweep')
adv_result('floor_reproduces', f'b_otsu and b_med equal floors.csv on all {N_ROIS} ROIs', 'asserted')

for name, frame in (('per_run', PER_RUN), ('top30', TOP), ('floors', FLOORS), ('contrasts', CON), ('arm_summary', ARM_SUMMARY), ('ladder', LADDER_TAB), ('domain', DOMAIN_TAB), ('safety', SAFETY), ('decisions', DECISIONS), ('adversarial_log', pd.DataFrame(ADV.values()))):
    frame.to_csv(OUT[name], index=False)
POOL.to_csv(OUT['pool'], index=False, compression='gzip')
print('\nwritten:', *OUT.values(), sep='\n  ')
print(f'total {(time.time() - NB_T0) / 60:.1f} min' + (' [SMOKE RUN: not a result]' if SMOKE_ROIS is not None else ''))


                                                             item  result                                                                                                                                                                                                             detail
plateau (L050 and L150 non-inferior to A3 on P@10, P@20 and pool)   False                                                                                                                                                          plateau fails on L050: report and stop; no post hoc level
       near-glass floor suffices (L025 non-inferior on all three)   False                                                                                                                                                                    reported only; the level is not moved in-sample
             alternative estimator W128 non-inferior on all three    True                                                                        


written:
  ../../results/tm_floor_definition_49roi_6seed_per_run.csv
  ../../results/tm_floor_definition_49roi_6seed_top30.csv
  ../../results/tm_floor_definition_49roi_6seed_floors.csv
  ../../results/tm_floor_definition_49roi_6seed_contrasts.csv
  ../../results/tm_floor_definition_49roi_6seed_arm_summary.csv
  ../../results/tm_floor_definition_49roi_6seed_ladder.csv
  ../../results/tm_floor_definition_49roi_6seed_domain.csv
  ../../results/tm_floor_definition_49roi_6seed_safety.csv
  ../../results/tm_floor_definition_49roi_6seed_decisions.csv
  ../../results/tm_floor_definition_49roi_6seed_adversarial_log.csv
  ../../results/tm_floor_definition_49roi_6seed_pool.csv.gz
total 77.7 min


## Reading

Written 2026-10-02, after the full run (77.7 min). Every gate held:
- **Parity.** `A3` re-searched on all 294 clicks and `A1` on 49 reproduce the recorded pools exactly, along with their
  thresholds, templates and TP@K.
- **Verbatim copies.** The 18 search helpers and 9 statistics functions are verbatim.
- **Floors.** The floors reproduce `floors.csv`.
- **Statistics.** The statistics reproduce the recorded `A3 - A1` rows.
- **Hand checks.** The hand check (`L050 - A3`, K = 10) and the pool re-count agree.

The setup is the band-floored harness, ranked by `tm_score`. Every number is **in-sample** on the 49 ROIs.

### Decision-rule outcome

| pre-registered item | result |
|---|---|
| Plateau: `L050` and `L150` non-inferior to `A3` on P@10, P@20 and pool | **Fails on the low side.** `L050` is a significant loss on all three: P@10 -1.39 pts [-2.28, -0.58], P@20 -1.17 [-1.77, -0.61], pool -0.72 mitoses per click [-0.97, -0.48]; Holm <= 0.021. `L150` is non-inferior on all three. |
| Level | **c = 1.0 (`b_otsu`) stays.** It is the only level validated against `A1`, and the rule forbids choosing a level in-sample. The sweep did **not** show that c = 1.0 is the centre of a plateau; it is not. |
| `L025` (low floor) non-inferior | **No.** It is a significant loss on all three. |
| `ALT` = `W128`, chosen before the search, non-inferior | **Yes, on all three, with tight CIs.** P@10 +0.24 [-0.10, +0.58], P@20 +0.05 [-0.15, +0.26], pool +0.04 [-0.01, +0.11]. The ROI-wide Otsu is not special. |
| `L200` non-inferior | P@20 and pool yes. P@10 no: the point estimate is 0.00, but the CI's lower bound is -1.33. |
| Zig-zag | none |
| Domain flags | none |
| "Degenerate templates: True" | **Mislabelled by my flag.** All 11 flags come from 3 seeds (`301` s2, `284` s3, `245` s2) whose *unclipped* `A0` template already has mass weight <= 0. None is caused by the floor; the minimum template SD is 0.005 even at `L200`. |

### The ladder

| floor | P@10 | P@20 | pool mitoses per click | white-adjacent FP@10 | other FP@10 | M2 | clicks better / worse than no clip (TP@10) | mean worst click (TP@10) |
|---|---|---|---|---|---|---|---|---|
| none (`A1`) | 54.6 | 47.2 | 24.68 | 316 | 1019 | 0.237 | - | 4.16 |
| 0.25x | 55.4 | 47.7 | 24.99 | 300 | 1010 | 0.229 | 27 / 4 | 4.31 |
| 0.5x | 56.5 | 48.7 | 25.49 | 277 | 1002 | 0.217 | 58 / 6 | 4.39 |
| **1.0x (`A3`)** | **57.9** | **49.9** | **26.21** | 240 | 998 | 0.194 | 95 / 20 | 4.55 |
| 1.5x | 58.2 | 50.2 | 26.63 | 211 | 1019 | 0.172 | 106 / 38 | 4.76 |
| 2.0x | 57.9 | 50.0 | 26.67 | 209 | 1029 | 0.169 | 116 / 58 | 4.63 |
| `W128` | 58.1 | 49.9 | 26.25 | 232 | 999 | 0.188 | 97 / 21 | 4.63 |

Adjacent steps at K = 10, in points:

| step | change | 95 % CI |
|---|---|---|
| none -> 0.25x | +0.85 | [+0.41, +1.36] |
| 0.25x -> 0.5x | +1.05 | [+0.65, +1.50] |
| 0.5x -> 1.0x | +1.39 | [+0.58, +2.28] |
| 1.0x -> 1.5x | +0.27 | [-0.61, +1.09] |
| 1.5x -> 2.0x | -0.27 | [-0.95, +0.41] |

The ladder is monotone up to 1.0x and flat after it, with no zig-zag.

### What it means for carrying over

1. **The response to the floor level is asymmetric.** The gain rises steeply up to 1.0x `b_otsu`, then is flat in
   mean from 1.0x to 2.0x. `b_otsu` sits at or just past the knee. With no 0.75x arm, the knee cannot be placed more
   precisely.
2. **Undershooting costs; overshooting does not, in mean.**
   - An estimator landing at half the level loses about 40 % of the P@10 gain (1.39 of 3.30 pts) and about half the
     pool gain (0.72 of 1.52).
   - Up to 1.5x costs nothing in mean.
3. **Overshooting is not free per click.** From 1.0x to 2.0x the mean is flat only because gains grow alongside the
   losses:
   - clicks worse than no clip go from 20 to 38 to 58;
   - clicks losing 2 or more TP go from 0 to 2 to 6.
   - *Post hoc:* among the levels that reach the plateau mean, 1.0x has the fewest losing clicks. That is a reading,
     not the reason c = 1.0 stays.
4. **The estimator.** A floor from a 128 µm window around the click (`W128`) is non-inferior to the ROI-wide floor, so
   the floor does not need the whole ROI.
   - Caveat: `W128` lands at 0.89-1.23x `b_otsu` (5th to 95th percentile), mostly on the flat side. It does not test a
     larger downward spread.
5. **The stress test now has a concrete target.** Under paler haematoxylin, eosin bleed, gamma and compression, measure
   where `b_otsu` and the window estimator land relative to this knee, not just P@K. An estimator that falls well below
   it on another lab's slides would lose the gain.

### Mechanism

**A low floor is not enough.**
- `L025` clips a median 13.5 % of ROI pixels, against a median ROI white share of 4.9 %. So it removes glass plus the
  palest tissue.
- It recovers only about 26 % of the P@10 gain (0.85 of 3.30) and 20 % of the pool gain.
- The nuisance is light ring content broadly, not glass specifically. M2 falls monotonically with the level, from 0.237
  to 0.169.

**My prediction P4 failed in form.**
- I predicted that precision would follow white-FP removal until it saturated. Instead P@10 saturates at 1.0x, while
  white-adjacent FP removal continues to 1.5x (240 -> 211).
- Above 1.0x, the removed white-adjacent FPs are replaced by other FPs (998 -> 1019 -> 1029), not by mitoses.
- That is not a rival explanation of the gain; it is its ceiling. Once light surroundings are flattened, a higher floor
  turns the score toward dense dark nuclei, which is the feature-ranking problem.

**The generator keeps growing to 1.5x.** Pool mitoses rise from 26.21 to 26.63 per click (+0.42, Holm 0.009). Those
are more mitoses in the pool, not in the top 20.

**Lumen-adjacent mitoses.** White-adjacent mitoses in the top 30 run 360 -> 351-354 across levels.

**Safety.**
- Within each arm, the share of the seed's kept box clipped to the floor shows no relation to the TP@10 change
  (Spearman -0.09 to +0.05, all p >= 0.11).
- No floor flattened a template.

### `L150`: the open trade-off, for the user to decide

**For 1.5x:**
- pool +0.42 mitoses per click (Holm 0.009);
- mean worst click 4.76 against 4.55, and zero-TP@10 clicks 9 against 11;
- P@10 / P@20 +0.27 / +0.34 (not significant);
- it sits mid-plateau, so it is more robust to an estimator that undershoots.

**Against 1.5x:**
- twice the clicks worse than no clip (38 against 20), and 2 clicks losing 2 or more;
- dips at P@10 in breast (-1.9) and melanoma (-1.2).

Moving the level needs held-out or stress-test data. This sweep was not allowed to choose it.

### Per domain (descriptive; 7 ROIs each resolves none of it)

- **Below 1.0x,** the losses concentrate in lung, lymphosarcoma, neuroendocrine and soft tissue: `L025` is -2.6 to -4.3
  at P@10. Breast, mast cell and melanoma barely move.
- **Above 1.0x,** lung and soft tissue gain (`L200` soft tissue +2.6), while breast and melanoma lose (`L200` -2.9 and
  -1.2 at P@10).

### What this cannot answer

- stain and scanner shift (the stress test);
- held-out ROIs;
- later rounds of the loop;
- chromatin od;
- where exactly the knee lies between 0.5x and 1.0x.
